<a href="https://colab.research.google.com/github/musorsey-web/lab/blob/main/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 3. Потоковая обработка сенсорных событий в Spark Structured Streaming

**Дисциплина:** «Методы обработки больших данных»

· **Этап пайплайна:** `Stream → Process`

**Среда:** Google Colab, PySpark 4.2.0, встроенный источник `rate` (Kafka не требуется)

**Максимум:** 10 баллов

· **Лекция:** № 3 «Потоковые данные и распределённая обработка событий»

## Поля для заполнения

| Поле | Значение |
|---|---|
| **Фамилия, имя, отчество** | |
| **Группа** | |
| **Вариант** | |
| **Дата выполнения** | |

*(заполните таблицу и те же значения продублируйте в ячейке 1 — они попадут в отчёт)*

## Критерии оценки

| Часть | Содержание | Баллы |
|---|---|---:|
| 1 | Поток и телеметрия с `event_time`, опоздавшие события по варианту | 2 |
| 2 | Окна, агрегаты и корректный снимок результата | 2 |
| 3 | Эксперимент с watermark: два значения и измеримая метрика | 2 |
| 4 | Перенос на Kafka: что меняется в коде | 1 |
| 5 | Педагогическая часть: `TEACH CARD` и фрагмент занятия | 3 |

## Как работать

1. **Файл → Сохранить копию на Диске.** Работайте в своей копии.
2. В ячейке 1 укажите ФИО, группу и свой вариант. Параметры варианта считаются формулой —
   править их вручную нельзя.
3. Выполняйте ячейки сверху вниз. Места с пометкой `TODO` дописываете вы; до этого ячейка
   честно падает с `NotImplementedError`.
4. Поля вида **«Ваш ответ:»** и пустые ячейки таблиц заполняются текстом — они оцениваются
   так же, как код.
5. Перед сдачей запустите самопроверку в конце ноутбука.

> Все выводы должны опираться на **ваши** числа. Прогон `rate`-источника зависит от
> производительности среды, поэтому значения у всех разные — это нормально.

## Теоретический минимум

| Понятие | Коротко |
|---|---|
| `event_time` | когда событие физически произошло (ось анализа) |
| `processing_time` | когда движок его обработал |
| окно $W/S$ | интервал длиной $W$, повторяющийся через $S$; одно событие попадает примерно в $M \approx W/S$ окон |
| watermark $\Delta$ | $\mathrm{WM} = \max(t_{event}) - \Delta$; окна позади watermark финализируются, их состояние удаляется |
| состояние | «(робот, окно) → счётчики»; растёт с числом живых окон |

Компромисс, который вы измерите в части 3: **больше $\Delta$ — полнее данные, но больше
состояния и позже готов итог.**

Подробности — в лекции № 3 и в `README_lab_03.md`.

---
# Часть 1. Поток и телеметрия (2 балла)

## Ячейка 1. Вариант и параметры (дана готовой)

Параметры вычисляются формулой: группа `(n−1)//5` задаёт нагрузку, число роботов и частоту
опоздавших событий, позиция `(n−1)%5` — геометрию окна, сдвиг опоздавших событий,
дополнительную метрику и аудиторию для методической части. Заполните три строки вверху.

In [11]:
# !pip -q install "pyspark==4.2.0"      # в Colab раскомментируйте эту строку
import json
import time
from typing import Dict, List

# =========== ЗАПОЛНИТЕ ===========
STUDENT_NAME  = "Пушкин Елисей Викторович"      # <-- ваши ФИО
STUDENT_GROUP = "Индор-221"                   # <-- ваша группа
VARIANT       = 19                          # <-- ваш вариант
# =================================
assert 1 <= VARIANT <= 25, "Вариант — целое число от 1 до 25"


def variant_params(n: int) -> dict:
    """Параметры варианта по таблице задания.

    Группа вариантов (n−1)//5 задаёт нагрузку и число роботов,
    позиция внутри группы (n−1)%5 — геометрию окна, долю и сдвиг опоздавших событий.
    """
    rows_per_sec = [20, 30, 40, 50, 60][(n - 1) // 5]
    robots       = [3, 4, 5, 6, 8][(n - 1) // 5]
    late_every   = [11, 17, 19, 23, 29][(n - 1) // 5]     # каждое k-е событие опаздывает
    window_s, slide_s = [(10, 5), (20, 5), (20, 10), (30, 10), (15, 5)][(n - 1) % 5]
    late_shift_s = [15, 20, 25, 30, 35][(n - 1) % 5]      # на сколько сдвинут event_time
    extra_metric = ["vibration_rms", "temp_spread", "temp_stddev",
                    "hot_share", "vibration_p95"][(n - 1) % 5]
    audience     = ["5–7 класс", "8–9 класс", "10–11 класс",
                    "кружок робототехники", "СПО"][(n - 1) % 5]
    return {
        "rows_per_sec": rows_per_sec,
        "robots": robots,
        "window_s": window_s,
        "slide_s": slide_s,
        "late_every": late_every,
        "late_shift_s": late_shift_s,
        "watermark_short": "10 seconds",
        "watermark_long": f"{late_shift_s + 10} seconds",   # заведомо больше сдвига
        "extra_metric": extra_metric,
        "audience": audience,
    }


CFG = variant_params(VARIANT)
RUN_SECONDS = 15        # длительность одного прогона потока

print(f"Студент : {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант : {VARIANT}")
for key, value in CFG.items():
    print(f"  {key:<16}: {value}")
print(f"\nОкон на одно событие (теория): M ≈ W/S = "
      f"{CFG['window_s']}/{CFG['slide_s']} = {CFG['window_s'] / CFG['slide_s']:.0f}")
print(f"Ожидаемая нагрузка: {CFG['rows_per_sec']} событий/с, "
      f"доля опоздавших ≈ {100 / CFG['late_every']:.1f} %")



def todo(step):
    raise NotImplementedError(f"Не выполнен шаг TODO {step} — допишите код в отмеченном месте")

Студент : Пушкин Елисей Викторович, группа Индор-221
Вариант : 19
  rows_per_sec    : 50
  robots          : 6
  window_s        : 30
  slide_s         : 10
  late_every      : 23
  late_shift_s    : 30
  watermark_short : 10 seconds
  watermark_long  : 40 seconds
  extra_metric    : hot_share
  audience        : кружок робототехники

Окон на одно событие (теория): M ≈ W/S = 30/10 = 3
Ожидаемая нагрузка: 50 событий/с, доля опоздавших ≈ 4.3 %


## Ячейка 2. Сессия Spark

**TODO 2.** Создайте `SparkSession` в локальном режиме.

Подсказки: мастер `local[*]` (потоку нужно минимум два потока выполнения — один занимает
приёмник источника); `spark.sql.shuffle.partitions = 4`, иначе Spark создаст 200 почти пустых
партиций и каждый micro-batch будет длиться лишние секунды; `setLogLevel("ERROR")` уберёт шум
в выводе. В конце напечатайте версию Spark и число ядер.

In [9]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName(f"Lab03_StructuredStreaming_v{VARIANT}")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.metricsEnabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)
print("Ядер    :", spark.sparkContext.defaultParallelism)
print("Shuffle :", spark.conf.get("spark.sql.shuffle.partitions"))

Spark   : 4.0.4
Master  : local[*]
Ядер    : 2
Shuffle : 4


## Ячейка 3. Телеметрия робота

**TODO 3.** Напишите функцию `build_telemetry(source, cfg)`, которая превращает поток `rate`
(колонки `timestamp`, `value`) в телеметрию:

| Колонка | Как считается |
|---|---|
| `robot_id` | `"robot-" + (value % cfg["robots"])` |
| `is_late` | `value % cfg["late_every"] == 0` — «правда» эксперимента: какое событие опоздало |
| `event_time` | для опоздавших `timestamp − cfg["late_shift_s"] секунд`, иначе `timestamp` |
| `motor_temp` | `43.0 + (value % 25) * 0.55` |
| `vibration` | `abs(sin(value / 8))` |

Сначала проверьте логику на **конечной** таблице `spark.range(0, 12)` с колонкой
`timestamp = current_timestamp()`: в потоке искать ошибку в выражении гораздо дороже.
Напечатайте таблицу и число опоздавших событий в примере.

**Важно:** сдвигать нужно `event_time` в **прошлое**. Событие «из будущего» watermark не
отбросит, и эксперимент части 3 не получится.

In [12]:
def build_telemetry(source, cfg: dict):
    """Превратить сырой поток rate в телеметрию роботов.

    Логика одна и та же для потока и для статической таблицы, поэтому функция
    принимает DataFrame: так её можно отладить на конечных данных.
    """
    return (
        source
        .withColumn("robot_id", F.concat(F.lit("robot-"),
                                         (F.col("value") % cfg["robots"]).cast("string")))
        .withColumn("is_late", F.col("value") % cfg["late_every"] == 0)
        .withColumn("event_time",
                    F.when(F.col("is_late"),
                           F.col("timestamp") - F.expr(f"INTERVAL {cfg['late_shift_s']} SECONDS"))
                    .otherwise(F.col("timestamp")))
        .withColumn("motor_temp", F.lit(43.0) + (F.col("value") % 25).cast("double") * 0.55)
        .withColumn("vibration", F.abs(F.sin(F.col("value") / 8.0)))
        .select("robot_id", "event_time", "is_late", "motor_temp", "vibration")
    )


# Отладка логики на конечной таблице: те же выражения, но без потока.
preview_source = (spark.range(0, 12)
                  .withColumnRenamed("id", "value")
                  .withColumn("timestamp", F.current_timestamp()))
preview = build_telemetry(preview_source, CFG)
preview.show(12, truncate=False)
print("Опоздавших в примере:", preview.filter("is_late").count(), "из", preview.count())

+--------+--------------------------+-------+----------+-------------------+
|robot_id|event_time                |is_late|motor_temp|vibration          |
+--------+--------------------------+-------+----------+-------------------+
|robot-0 |2026-10-24 06:05:43.795117|true   |43.0      |0.0                |
|robot-1 |2026-10-24 06:06:13.795117|false  |43.55     |0.12467473338522769|
|robot-2 |2026-10-24 06:06:13.795117|false  |44.1      |0.24740395925452294|
|robot-3 |2026-10-24 06:06:13.795117|false  |44.65     |0.36627252908604757|
|robot-4 |2026-10-24 06:06:13.795117|false  |45.2      |0.479425538604203  |
|robot-5 |2026-10-24 06:06:13.795117|false  |45.75     |0.5850972729404622 |
|robot-0 |2026-10-24 06:06:13.795117|false  |46.3      |0.6816387600233341 |
|robot-1 |2026-10-24 06:06:13.795117|false  |46.85     |0.7675435022360271 |
|robot-2 |2026-10-24 06:06:13.795117|false  |47.4      |0.8414709848078965 |
|robot-3 |2026-10-24 06:06:13.795117|false  |47.95     |0.9022675940990952 |

---
# Часть 2. Окна, агрегаты и снимок (2 балла)

## Ячейка 4. Запуск потока с окнами

**TODO 4.** Напишите функцию `run_stream(cfg, watermark, seconds, name)`:

1. источник `rate` с `rowsPerSecond = cfg["rows_per_sec"]` и `numPartitions = 2`;
2. `build_telemetry`;
3. `withWatermark("event_time", watermark)`;
4. группировка по `window(event_time, "{W} seconds", "{S} seconds")` и `robot_id`;
5. агрегаты: `events` (счёт), `late_events` (сумма `is_late`), `avg_temp`, `max_temp` и
   дополнительная метрика вашего варианта (`cfg["extra_metric"]`);
6. запись в `memory`-приёмник, `outputMode("update")`, триггер 1 секунда;
7. ожидание `seconds` секунд, сохранение `list(query.recentProgress)`, затем `query.stop()`.

**Не используйте `query.processAllAvailable()`.** Источник `rate` бесконечен: «все доступные
данные» у него не заканчиваются, и вызов может заблокировать ячейку надолго. Останавливайте
поток по таймеру.

Функция возвращает `(таблица обновлений, список метрик батчей)`.

Выражения дополнительных метрик:

```python
AGGREGATES = {
    "vibration_rms": F.sqrt(F.avg(F.pow("vibration", 2))).alias("extra_metric"),
    "temp_spread":   (F.max("motor_temp") - F.min("motor_temp")).alias("extra_metric"),
    "temp_stddev":   F.stddev_samp("motor_temp").alias("extra_metric"),
    "hot_share":     F.avg((F.col("motor_temp") > 50).cast("double")).alias("extra_metric"),
    "vibration_p95": F.percentile_approx("vibration", 0.95).alias("extra_metric"),
}
```

In [13]:
AGGREGATES = {
    "vibration_rms": F.sqrt(F.avg(F.pow("vibration", 2))).alias("extra_metric"),
    "temp_spread":   (F.max("motor_temp") - F.min("motor_temp")).alias("extra_metric"),
    "temp_stddev":   F.stddev_samp("motor_temp").alias("extra_metric"),
    "hot_share":     F.avg((F.col("motor_temp") > 50).cast("double")).alias("extra_metric"),
    "vibration_p95": F.percentile_approx("vibration", 0.95).alias("extra_metric"),
}


def run_stream(cfg: dict, watermark: str, seconds: int = RUN_SECONDS, name: str = "robot_windows"):
    """Запустить поток на заданное время и вернуть (снимок результата, метрики прогресса).

    Остановка по таймеру, а не processAllAvailable(): источник rate порождает данные
    непрерывно, и «дождаться всех доступных данных» у него нельзя — метод может
    блокировать выполнение надолго.
    """
    source = (spark.readStream.format("rate")
              .option("rowsPerSecond", cfg["rows_per_sec"])
              .option("numPartitions", 2)
              .load())
    telemetry = build_telemetry(source, cfg)
    windowed = (
        telemetry
        .withWatermark("event_time", watermark)
        .groupBy(F.window("event_time", f"{cfg['window_s']} seconds",
                          f"{cfg['slide_s']} seconds"), "robot_id")
        .agg(F.count("*").alias("events"),
             F.sum(F.col("is_late").cast("int")).alias("late_events"),
             F.avg("motor_temp").alias("avg_temp"),
             F.max("motor_temp").alias("max_temp"),
             AGGREGATES[cfg["extra_metric"]])
    )
    query = (windowed.writeStream
             .format("memory").queryName(name)
             .outputMode("update")
             .trigger(processingTime="1 second")
             .start())
    started = time.time()
    while time.time() - started < seconds:
        time.sleep(1)
    progress = list(query.recentProgress)
    query.stop()
    return spark.sql(f"SELECT * FROM {name}"), progress


raw_updates, progress_short = run_stream(CFG, CFG["watermark_short"], name="w_short")
print(f"Строк в memory-sink: {raw_updates.count()}")
print(f"Уникальных пар (окно, робот): "
      f"{raw_updates.select('window', 'robot_id').distinct().count()}")

Строк в memory-sink: 220
Уникальных пар (окно, робот): 42


**Ваш ответ (0,5 балла части 2).** Почему эти два числа различаются? Что именно хранит
`memory`-приёмник в режиме `update`?

> *(первое считает все строки записанные в таблицу за время работы, а второе только уникальные пары)*

## Ячейка 5. Снимок состояния окон

**TODO 5.** Постройте функцию `snapshot(updates)`, которая оставляет по одной строке на пару
«окно — робот» — ту, где число событий максимально (это и есть последнее состояние окна).

Подсказка: оконная функция
`row_number().over(Window.partitionBy("window", "robot_id").orderBy(F.col("events").desc()))`,
затем фильтр `rn = 1`. Выведите `window_start`, `window_end`, `robot_id`, `events`,
`late_events`, округлённые `avg_temp`, `max_temp` и вашу дополнительную метрику, отсортировав
по убыванию начала окна.

In [15]:
def snapshot(updates):
    """Снимок последнего состояния каждого окна.

    В режиме update memory-sink хранит ИСТОРИЮ обновлений: одна и та же пара
    (окно, робот) появляется после каждого micro-batch, в котором она изменилась.
    Итоговое состояние — строка с максимальным числом событий.
    """
    ranked = updates.withColumn(
        "rn", F.row_number().over(
            __import__("pyspark").sql.Window
            .partitionBy("window", "robot_id").orderBy(F.col("events").desc())))
    return (ranked.filter("rn = 1").drop("rn")
            .select(F.col("window.start").alias("window_start"),
                    F.col("window.end").alias("window_end"),
                    "robot_id", "events", "late_events",
                    F.round("avg_temp", 2).alias("avg_temp"),
                    F.round("max_temp", 2).alias("max_temp"),
                    F.round("extra_metric", 4).alias(CFG["extra_metric"]))
            .orderBy(F.col("window_start").desc(), "robot_id"))


result_short = snapshot(raw_updates)
result_short.show(12, truncate=False)
print("Окон в снимке:", result_short.count())
print("Событий учтено в окнах:", result_short.agg(F.sum("events")).first()[0])

+-------------------+-------------------+--------+------+-----------+--------+--------+---------+
|window_start       |window_end         |robot_id|events|late_events|avg_temp|max_temp|hot_share|
+-------------------+-------------------+--------+------+-----------+--------+--------+---------+
|2026-10-24 06:07:10|2026-10-24 06:07:40|robot-0 |47    |0          |49.79   |56.2    |0.4894   |
|2026-10-24 06:07:10|2026-10-24 06:07:40|robot-1 |47    |0          |49.73   |56.2    |0.4894   |
|2026-10-24 06:07:10|2026-10-24 06:07:40|robot-2 |45    |0          |49.73   |56.2    |0.4889   |
|2026-10-24 06:07:10|2026-10-24 06:07:40|robot-3 |46    |0          |49.73   |56.2    |0.5      |
|2026-10-24 06:07:10|2026-10-24 06:07:40|robot-4 |47    |0          |49.61   |56.2    |0.4894   |
|2026-10-24 06:07:10|2026-10-24 06:07:40|robot-5 |47    |0          |49.85   |56.2    |0.5106   |
|2026-10-24 06:07:00|2026-10-24 06:07:30|robot-0 |104   |0          |49.8    |56.2    |0.4808   |
|2026-10-24 06:07:00

---
# Часть 3. Метрики и эксперимент с watermark (2 балла)

## Ячейка 6. Метрики micro-batch

**TODO 6.** Напишите `progress_summary(progress)`, которая по списку `recentProgress`
возвращает словарь: число батчей с данными, суммарное число входных событий, пропускную
способность (события / суммарное время `durationMs["triggerExecution"]`), среднюю длительность
батча, максимальное число строк состояния и память состояния
(`stateOperators[0]["numRowsTotal"]`, `["memoryUsedBytes"]`) и, главное, суммарное
`numRowsDroppedByWatermark` — число событий, отброшенных по watermark.

Выведите сводку и блок `progress[-1]["eventTime"]`: в нём видно, на сколько watermark отстаёт
от максимального времени события.

In [16]:
def progress_summary(progress: List[dict]) -> dict:
    """Свести метрики micro-batch в показатели, обсуждавшиеся на лекции."""
    batches = [p for p in progress if p.get("numInputRows", 0) > 0]
    state = [p["stateOperators"][0] for p in batches if p.get("stateOperators")]
    return {
        "батчей": len(batches),
        "входных событий": sum(p["numInputRows"] for p in batches),
        "throughput, соб./с": round(sum(p["numInputRows"] for p in batches)
                                    / max(sum(p["durationMs"]["triggerExecution"]
                                              for p in batches) / 1000, 1e-9), 1),
        "средняя длительность батча, мс": round(sum(p["durationMs"]["triggerExecution"]
                                                    for p in batches) / len(batches)),
        "строк состояния (макс.)": max((s["numRowsTotal"] for s in state), default=0),
        "память состояния, КиБ": round(max((s["memoryUsedBytes"] for s in state), default=0) / 1024, 1),
        "событий отброшено watermark": sum(s.get("numRowsDroppedByWatermark", 0) for s in state),
    }


summary_short = progress_summary(progress_short)
print(f"Watermark {CFG['watermark_short']}\n")
for key, value in summary_short.items():
    print(f"  {key:<32}: {value}")

last = progress_short[-1]
print("\nГраницы времени последнего батча:")
for key, value in last["eventTime"].items():
    print(f"  {key:<12}: {value}")

Watermark 10 seconds

  батчей                          : 9
  входных событий                 : 650
  throughput, соб./с              : 56.7
  средняя длительность батча, мс  : 1274
  строк состояния (макс.)         : 36
  память состояния, КиБ           : 15.0
  событий отброшено watermark     : 30

Границы времени последнего батча:
  avg         : 2026-10-24T06:07:14.149Z
  max         : 2026-10-24T06:07:15.839Z
  min         : 2026-10-24T06:06:45.279Z
  watermark   : 2026-10-24T06:07:04.839Z


## Ячейка 7. Два значения watermark

**TODO 7.** Повторите прогон с `CFG["watermark_long"]` (он заведомо больше сдвига опоздавших
событий вашего варианта) и сведите оба результата в таблицу: входные события, пропускная
способность, длительность батча, строки и память состояния, отброшенные события, число окон в
снимке и сумма `late_events`.

In [17]:
raw_long, progress_long = run_stream(CFG, CFG["watermark_long"], name="w_long")
result_long = snapshot(raw_long)
summary_long = progress_summary(progress_long)

comparison = [
    {"watermark": CFG["watermark_short"], **summary_short,
     "окон в снимке": result_short.count(),
     "опоздавших событий в окнах": int(result_short.agg(F.sum("late_events")).first()[0] or 0)},
    {"watermark": CFG["watermark_long"], **summary_long,
     "окон в снимке": result_long.count(),
     "опоздавших событий в окнах": int(result_long.agg(F.sum("late_events")).first()[0] or 0)},
]
import pandas as pd
print(pd.DataFrame(comparison).set_index("watermark").T.to_string())

watermark                       10 seconds  40 seconds
батчей                                 9.0        12.0
входных событий                      650.0       650.0
throughput, соб./с                    56.7        57.9
средняя длительность батча, мс      1274.0       936.0
строк состояния (макс.)               36.0        46.0
память состояния, КиБ                 15.0        19.8
событий отброшено watermark           30.0         0.0
окон в снимке                         42.0        46.0
опоздавших событий в окнах            57.0        87.0


**Ваш ответ (1 балл части 3).** Заполните по своим числам:

| Показатель | watermark короткий | watermark длинный |
|---|---:|---:|
| событий отброшено watermark |30| 0|
| строк состояния (максимум) |36 |46 |
| опоздавших событий учтено в окнах |57 |87|

1. Что изменилось при увеличении watermark и **чем за это заплатили**?
   > *(стали учитываться все данные и резултатов стало больше)*
2. Какой эффект этот прогон измерить **не позволяет** и почему?
   > *(используется больше памяти )*

## Ячейка 8. Кратность окон

**TODO 8.** Сравните фактическое число попаданий событий в окна (сумма `events` в снимке) с
числом входных событий. Отношение должно быть близко к $M \approx W/S$ вашего варианта.
Напечатайте оба числа и объясните расхождение.

In [18]:
events_total = summary_short["входных событий"]
rows_in_windows = result_short.agg(F.sum("events")).first()[0] or 0
multiplicity = rows_in_windows / max(events_total, 1)

print(f"Событий на входе за прогон        : {events_total}")
print(f"Суммарно учтено в окнах           : {rows_in_windows}")
print(f"Фактическая кратность M           : {multiplicity:.2f}")
print(f"Теоретическая кратность W/S       : {CFG['window_s'] / CFG['slide_s']:.0f}")
print("\nРасхождение объясняется краевыми окнами: события начала и конца прогона\n"
      "попадают не во все W/S окон, а часть окон ещё не финализирована.")

Событий на входе за прогон        : 650
Суммарно учтено в окнах           : 1920
Фактическая кратность M           : 2.95
Теоретическая кратность W/S       : 3

Расхождение объясняется краевыми окнами: события начала и конца прогона
попадают не во все W/S окон, а часть окон ещё не финализирована.


**Ваш ответ.** Фактическая кратность 2.95 при теоретической 3. Расхождение
объясняется:

> *(обытия начала и конца прогона
попадают не во все W/S окон, а часть окон ещё не финализирована.)*

---
# Часть 4. Перенос на Kafka (1 балл)

Код ниже **не выполняется**: broker в работе не нужен. Разберите его и ответьте на вопросы.

In [19]:
kafka_code = """
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "robot.telemetry")
    .option("startingOffsets", "latest")
    .option("maxOffsetsPerTrigger", 5000)
    .load()
)

schema = "robot_id STRING, event_time TIMESTAMP, motor_temp DOUBLE, vibration DOUBLE"
telemetry = (
    raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
       .select("e.*")
)
"""
print(kafka_code)


raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "robot.telemetry")
    .option("startingOffsets", "latest")
    .option("maxOffsetsPerTrigger", 5000)
    .load()
)

schema = "robot_id STRING, event_time TIMESTAMP, motor_temp DOUBLE, vibration DOUBLE"
telemetry = (
    raw.select(F.from_json(F.col("value").cast("string"), schema).alias("e"))
       .select("e.*")
)



**Ваши ответы (1 балл).**

| Вопрос | Ваш ответ |
|---|---|
| Какие строки вашего кода нужно заменить при переходе на Kafka и какие остаются без изменений? |изменяется: spark.readStream.format("kafka"), не меняется: writeStream.format("memory") |
| Что такое offset и почему это **не** временная метка? |тк это порядковый номер, а в одно время может прийти 2 значения с датчка, поэтому позиция не равно время |
| Как `key = robot_id` влияет на порядок событий и на параллелизм? | - |
| Зачем в потоковом запросе с Kafka обязателен `checkpointLocation`? |что бы считывать новые пакеты сразу, а не начинать снова с тех которые уже обработали|

---
# Часть 5. Педагогическая часть (3 балла)

Инженерный результат работы — три утверждения, которые можно объяснить школьнику без слов
«watermark» и «micro-batch»:

1. **У события есть два времени** — когда оно случилось и когда о нём узнали; считать надо по
   первому.
2. **Ждать опоздавших — значит занимать память**: чем дольше система ждёт, тем больше хранит и
   тем позже выдаёт итог.
3. **Скользящее окно считает одно событие несколько раз** — ровно $W/S$ раз, и это свойство
   метода, а не ошибка.

## 5.1. `TEACH CARD` (1,5 балла)

Аудитория задана вашим вариантом (`CFG["audience"]`), числа берутся из **вашего** прогона.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** |Потоковая обработка данных с датчиков робота |
| **Целевая аудитория** |кружок робототехники |
| **Исследовательский вопрос** |почему увеличение времени ожидания опоздавших данных делает итоговый отчёт точнее, но заставляет компьютер тратить больше оперативной памяти? |
| **Источник данных** |Искусственный поток данных от 6 роботов, генерируемый со скоростью 50 событий в секунду |
| **Инженерная концепция, сохраняемая без упрощения** |У события есть два времени когда оно произошло в реальности и когда система о нём узнала. Система должна ждать опоздавших, но за это платит памятью и задержкой выдачи результата |
| **Что упрощается относительно университетской лабораторной** |вместо реального Кафка используется встроенный генератор рейт, готовый код учащийся меняет только числа и наблюдает результат. |
| **Алгоритм действий обучающегося (4–6 шагов)** |1) Запустить генерацию потока событий и посмотреть, как они попадают во временные окна 2) Запустить обработку с коротким временем ожидания и посчитать, сколько событий система упустила 3) Повторить обработку с длинным временем ожидания и сравнить числа 4) Заполнить таблицу сколько событий отброшено, сколько строк состояния, сколько окон получилось 5) Сделать вывод что изменилось и сколько ресурсов было затрачено на это |
| **Измеримый результат / метрика** |Число отброшенных событий, число строк состояния, число учтённых опоздавших событий |
| **Критерий успешного выполнения** |Учащийся показал: при увеличении времени ожидания отброшенных событий становится 0, но строк состояния и памяти тратится больше.|
| **Вариант усложнения для НТО / хакатона** |Подключить реальный источник данных |
| **Риски и ограничения** |Генератор rate даёт идеальные данные без сетевых задержек, поэтому в реальности потери будут больше |

## 5.2. Фрагмент занятия на 15–20 минут (1,5 балла)

Выберите **один** из трёх выводов и опишите фрагмент. Обязательны: действия учеников (а не
рассказ учителя), **два числа вашего варианта**, ключевой вопрос и способ проверки понимания.

| Пункт | Ваш ответ |
|---|---|
| Вывод (1, 2 или 3) |ожидание опоздавших данных повышает полноту отчета, но увеличивает расход оперативной памяти и задерживает выдачу итогового результата |
| Класс и формат |10–11 класс, практическое занятие|
| Время |20 минут|
| Материалы |Google Colab, распечатанный лист с пустой сравнительной таблицей, презентация |
| Числа вашего прогона и как они используются |Используются конкретные метрики из лабораторной работы: при ожидании 10 секунд система отбросила 30 событий и заняла 15,0 КБ памяти. При ожидании 40 секунд отброшенных событий 0, но память выросла до 19,8 КБ. Ученики вносят эти числа в свои таблицы для количественного сравнения |

**Ход фрагмента (3–5 шагов, кто что делает):**
10 Постановка задачи: Учитель выводит на экран пустую сравнительную таблицю. Ученики в парах запускают ячейку кода с коротким временем ожидания и самостоятельно фиксируют в своих листах количество отброшенных событий и объем занятой памяти
2)Проведение эксперимента: Ученики изменяют параметр времени ожидания на 40 секунд, перезапускают ячейку и заполняют второй столбец таблицы своими полученными значениями Учитель контролирует корректность выполнения кода и помогает при возникновении технических ошибок
3)Анализ данных: Ученики рассчитывают разницу в потреблении памяти и количестве учтенных событий между двумя прогонами. Каждая пара формулирует короткий письменный вывод о том, какими ресурсами система потратила за отсутствие потерь данных
4) Обобщение: Учитель собирает ответы, фиксируется закономерность: увеличение времени ожидания обнуляет потери данных, но увеличивает нагрузку на память. Обсуждается применимость этого правила для реального робота, например риск переполнения памяти

**Ключевой вопрос для учеников: Если мы установим время ожидания в 1000 секунд, чтобы гарантированно не потерять ни одного показания датчика, какие два негативных последствия это вызовет в работе системы управления парком роботов?

**Проверка понимания (как узнаете, что вывод усвоен; какое заблуждение ловится):** ответ должен содержать указание на исчерпание оперативной памяти и критическую задержку в принятии решений

**Критерии.** 1,5 балла — `TEACH CARD` заполнен полностью, аудитория соответствует варианту,
инженерная концепция не искажена. 1,5 балла — фрагмент реализуем за 15–20 минут, содержит
действия учеников, числа своего прогона, ключевой вопрос и проверку понимания.

Не засчитывается: пересказ лекции без действий учеников; чужие числа; объяснение термина
«watermark» словом «watermark».

---
## Самопроверка перед сдачей

In [20]:
ANSWERS = {
    "dropped_short": None,      # событий отброшено watermark при коротком watermark
    "dropped_long": None,       # то же при длинном watermark
    "state_rows_short": None,   # строк состояния (максимум) при коротком
    "state_rows_long": None,    # то же при длинном
    "windows_short": None,      # окон в снимке при коротком watermark
    "multiplicity": None,       # фактическая кратность окон
}

checks = {
    "Заполнены ФИО и группа":        STUDENT_NAME != "Иванов Иван Иванович",
    "Вариант 1..25":                 1 <= VARIANT <= 25,
    "Телеметрия построена":          "raw_updates" in dir() and raw_updates.count() > 0,
    "Снимок окон получен":           "result_short" in dir() and result_short.count() > 0,
    "Снимок короче истории":         "result_short" in dir() and result_short.count() <= raw_updates.count(),
    "Метрики посчитаны":             "summary_short" in dir(),
    "Эксперимент с watermark":       all(ANSWERS[k] is not None for k in ("dropped_short", "dropped_long")),
    "Состояние измерено":            all(ANSWERS[k] is not None for k in ("state_rows_short", "state_rows_long")),
    "Кратность окон оценена":        ANSWERS["multiplicity"] is not None,
    "Длинный watermark даёт меньше потерь":
        (ANSWERS["dropped_long"] or 0) <= (ANSWERS["dropped_short"] or 0),
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")
print(f"\nВыполнено: {sum(checks.values())} из {len(checks)}")
print("\nПедагогическая часть (3 балла) проверяется преподавателем вручную.")

OK   Заполнены ФИО и группа
OK   Вариант 1..25
OK   Телеметрия построена
OK   Снимок окон получен
OK   Снимок короче истории
OK   Метрики посчитаны
НЕТ  Эксперимент с watermark
НЕТ  Состояние измерено
НЕТ  Кратность окон оценена
OK   Длинный watermark даёт меньше потерь

Выполнено: 7 из 10

Педагогическая часть (3 балла) проверяется преподавателем вручную.


## Перед отправкой

- [ ] заполнена таблица с ФИО, группой, вариантом и датой;
- [ ] `VARIANT` совпадает с номером в списке, параметры не правились руками;
- [ ] выполнены все ячейки, выводы сохранены;
- [ ] заполнены поля «Ваш ответ» в частях 2, 3 и 4;
- [ ] заполнен словарь `ANSWERS`;
- [ ] `TEACH CARD` заполнен полностью, аудитория соответствует варианту;
- [ ] фрагмент занятия содержит действия учеников, два числа вашего прогона, ключевой вопрос и
      проверку понимания;
- [ ] самопроверка показывает «Выполнено: 10 из 10».